In [1]:
import boto3
import json
import os
from dotenv import load_dotenv


# AWS Credentials
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION = os.getenv("AWS_REGION")


bedrock_runtime = boto3.client(
    service_name="bedrock-runtime",
    region_name=AWS_REGION,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY
)

MODEL_ID = "amazon.nova-micro-v1:0"

In [14]:
body = {
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "text": "what is the weather of Hyderabad today?"
                }
            ]
        }
    ],
    "inferenceConfig": {
        "max_new_tokens": 100,
        "temperature": 0.3
    }
}

response = bedrock_runtime.invoke_model(
    modelId=MODEL_ID,
    body=json.dumps(body),
    contentType="application/json",
    accept="application/json"
)

response_body = json.loads(response["body"].read())

final_response = response_body["output"]["message"]["content"][0]["text"]

print(f"Model Response : {final_response}")

Model Response : I don't have real-time data access capabilities. For the most accurate and up-to-date weather information for Hyderabad today, I recommend checking a reliable weather website or app, such as the Weather Channel, AccuWeather, or a local meteorological service. These sources provide current weather conditions, forecasts, and other relevant details.


In [ ]:
import requests

API_KEY = "OPENWEATHER_API_KEY"

def get_weather(city):

    url = (
        f"https://api.openweathermap.org/data/2.5/weather"
        f"?q={city}&appid={API_KEY}&units=metric"
    )

    response = requests.get(url)

    data = response.json()

    temp = data["main"]["temp"]

    description = data["weather"][0]["description"]

    return f"{temp}°C, {description}"

In [22]:
# from calculator_tool import calculator
def calculator(expression):

    return eval(expression)

In [26]:

print(calculator("20+30"))

50


In [28]:
def invoke(prompt):

    body = {
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "text": prompt
                    }
                ]
            }
        ],
        "inferenceConfig": {
            "max_new_tokens": 100,
            "temperature": 0.3
        }
    }

    response = bedrock_runtime.invoke_model(
        modelId=MODEL_ID,
        body=json.dumps(body),
        contentType="application/json",
        accept="application/json"
    )

    response_body = json.loads(
        response["body"].read()
    )

    return response_body["output"]["message"]["content"][0]["text"]

In [47]:
def choose_tool(question):

    prompt = f"""
You are a routing agent.

Available Tools:

1. weather
2. calculator

Question:
{question}

Return ONLY the answers:

weather
calculator
both
none
"""

    return invoke(prompt).strip().lower()

In [46]:
def extract_qus(question):
    prompt= f"""
Extract only the mathematical expression.

Question:
{question}

Return ONLY expression.

Examples:

What is 20+30?
20+30

Calculate 50*7
50*7
"""

    return invoke(prompt).strip()

In [45]:
def extract_city(question):

    prompt = f"""
Extract only the city name.

Question:
{question}

Return ONLY city name.

Examples:

What is weather in Hyderabad?
Hyderabad

Weather in Mumbai today
Mumbai
"""

    return invoke(prompt).strip()

In [49]:
def agent(question):

    decision = choose_tool(question)

    print("Selected Tool:", decision)

    if decision == "calculator":

        expression = extract_qus(question)

        print("Expression:", expression)

        result = calculator(expression)

    elif decision == "weather":

        city = extract_city(question)

        print("City:", city)

        result = get_weather(city)

    else:

        return "No suitable tool found"

    final_prompt = f"""
Question:
{question}

Tool Result:
{result}

Generate a helpful answer.
"""

    final_answer = invoke(final_prompt)

    return final_answer

In [50]:
print(agent("what is 20* 40"))

Selected Tool: calculator
Expression: 20*40
Certainly! When you multiply 20 by 40, you are essentially calculating the product of these two numbers. Here's a step-by-step explanation:

1. **Break Down the Numbers**:
   - 20 can be broken down as \(2 \times 10\).
   - 40 can be broken down as \(4 \times 10\).

2. **Use the Distributive Property**:
   - You can use the distributive property
